# Regrid E3SMv3 (ne30pg2) daily output to 1°×1° lat-lon, for Pyro SAI
Adapted from A. Duffey's `Regrid_E3SM_data.ipynb` (Feb 2026). **Fixes and changes:**
- **Missing comma** in `hilla_paths`: `'...sai.0201/.../' 's3://...ssp245_0101/...'` was concatenated into one invalid prefix, so those two runs were silently skipped.
- **Directory mismatch**: files were written to `./E3SMv3_HiLLA_regridded` but read back from `hilla_regridded/`. One `LOCAL_BASE_DIR` is now used throughout.
- `dirs[2:]` and `os.listdir(...)[0]` depended on arbitrary listing order, so they are replaced by explicit loops.
- `list_objects_v2` returns at most 1000 keys, so a paginator is used.
- Generalised to **several variables** (daily `TREFHTMX`, `TREFHT`, `RHREFHT`, `U10`, `PRECT`) and any table (`day`/`Amon`).
- Combined output is written as **Zarr** directly to `persistent-prod` (a 50-year daily global file is several GB). Existing outputs are skipped, and local temp files are removed after each file.

Afterwards, the regridded Zarr stores can be added to `pyrosai_catalog.json` as `{"kind": "zarr", "store": "s3://..."}` entries.

In [ ]:
import os, subprocess, shutil
from urllib.parse import urlparse
import boto3, s3fs, fsspec
import xarray as xr

# nco provides ncremap (needs ~8 GB RAM to install; image 5cb60c85bbb0 already ships cdo + nco)
if shutil.which("ncremap") is None:
    print("installing nco ..."); subprocess.run(["conda", "install", "-y", "-q", "-c", "conda-forge", "nco"], check=True)

In [ ]:
#########################  USER INPUTS  #########################
SRC_BUCKET_ROOT = "s3://reflective-persistent-prod-large/E3SMv3/G6-1.5K-HiLLA/"
CASES = [   # note the commas!
    "v3.LR.ssp245.g6_hilla.sai.0101",
    "v3.LR.ssp245.g6_hilla.sai.0151",
    "v3.LR.ssp245.g6_hilla.sai.0201",
    "v3.LR.ssp245_0101",
    "v3.LR.ssp245_0151",
    "v3.LR.ssp245_0201",
]
TABLE     = "day"          # check the bucket: daily output may be under 'day', 'Eday' or a stream name like 'h1'
VARIABLES = ["TREFHTMX", "TREFHT", "RHREFHT", "U10", "PRECT"]
MAP_FILE  = "../shared/Data/E3SM_maps/map_ne30pg2_to_cmip6_180x360_aave.20200201.nc"
LOCAL_BASE_DIR = "./E3SMv3_regrid_tmp"
DEST_ROOT = "s3://reflective-persistent-prod/<username>/E3SMv3_regridded"   # Zarr outputs
##################################################################
s3 = boto3.client("s3")
fs = s3fs.S3FileSystem()
os.makedirs(LOCAL_BASE_DIR, exist_ok=True)

def list_nc(prefix_url):
    p = urlparse(prefix_url); keys = []
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=p.netloc, Prefix=p.path.lstrip("/")):
        keys += [(p.netloc, o["Key"]) for o in page.get("Contents", []) if o["Key"].endswith(".nc")]
    return sorted(keys)

def run_ncremap(src, dst):
    r = subprocess.run(["ncremap", "-i", src, "-m", MAP_FILE, "-o", dst], capture_output=True, text=True)
    if r.returncode != 0:
        print("   ncremap failed:", r.stderr[-500:])
    return r.returncode == 0

# discover what actually exists before regridding anything
for case in CASES:
    found = fs.glob(f"{SRC_BUCKET_ROOT.rstrip('/')}/{case}/{TABLE}/*")
    print(case, "->", [f.split("/")[-1] for f in found] or f"nothing under {TABLE}/ (check TABLE)")

In [ ]:
for case in CASES:
    for var in VARIABLES:
        dest = f"{DEST_ROOT}/{case}/{TABLE}/{var}.zarr"
        if fs.exists(dest.replace("s3://", "") + "/.zmetadata") or fs.exists(dest.replace("s3://", "") + "/zarr.json"):
            print(f"[skip] {dest} exists"); continue
        keys = list_nc(f"{SRC_BUCKET_ROOT.rstrip('/')}/{case}/{TABLE}/{var}/")
        if not keys:
            print(f"[none] {case}/{TABLE}/{var}"); continue
        work = os.path.join(LOCAL_BASE_DIR, case, var); os.makedirs(work, exist_ok=True)
        print(f">>> {case} {var}: {len(keys)} files")
        first = True
        for bucket, key in keys:
            raw = os.path.join(work, os.path.basename(key))
            out = os.path.join(work, "rg_" + os.path.basename(key))
            s3.download_file(bucket, key, raw)
            ok = run_ncremap(raw, out)
            os.remove(raw)
            if not ok:
                raise RuntimeError(f"regrid failed for {key}")
            ds = xr.open_dataset(out, chunks={"time": 365})[[var] + [v for v in ("time_bnds", "time_bounds") if v in xr.open_dataset(out).variables]]
            ds = ds.chunk({"time": 365, "lat": -1, "lon": -1})
            for v in ds.variables:
                ds[v].encoding = {}
            ds.to_zarr(dest, mode="w" if first else "a", append_dim=None if first else "time", zarr_format=2)
            ds.close(); os.remove(out); first = False
            print(f"    appended {os.path.basename(key)}")
        shutil.rmtree(work)
        chk = xr.open_zarr(dest)
        print(f"    done: {dict(chk.sizes)}  {str(chk.time.values[0])[:10]} .. {str(chk.time.values[-1])[:10]}")

In [ ]:
# quick visual check of one output
ds = xr.open_zarr(f"{DEST_ROOT}/{CASES[0]}/{TABLE}/{VARIABLES[0]}.zarr")
ds[VARIABLES[0]].isel(time=0).plot()